In [12]:
import sys
print(sys.version)
help("modules request")


3.9.6 (default, Dec  2 2025, 07:27:58) 
[Clang 17.0.0 (clang-1700.6.3.2)]

Here is a list of modules whose name or summary contains 'request'.
If there are any, enter a module name to get more help.

urllib.request - An extensible library for opening URLs using a variety of protocols
pip._vendor.cachecontrol.controller - The httplib2 algorithms ported for use with requests.
pip._vendor.requests - Requests HTTP Library
pip._vendor.requests.__version__ 
pip._vendor.requests._internal_utils - requests._internal_utils
pip._vendor.requests.adapters - requests.adapters
pip._vendor.requests.api - requests.api
pip._vendor.requests.auth - requests.auth
pip._vendor.requests.certs - requests.certs
pip._vendor.requests.compat - requests.compat
pip._vendor.requests.cookies - requests.cookies
pip._vendor.requests.exceptions - requests.exceptions
pip._vendor.requests.help - Module containing bug report helper(s).
pip._vendor.requests.hooks - requests.hooks
pip._vendor.requests.models - requests.model

In [16]:
%pip install requests beautifulsoup4 "urllib3<2"

     |████████████████████████████████| 144 kB 7.5 MB/s eta 0:00:01
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.6.3
    Uninstalling urllib3-2.6.3:
      Successfully uninstalled urllib3-2.6.3
You should consider upgrading via the '/Users/jakefrischmann/repos/OrbitUMD/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import re
import time
import random
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

BASE = "https://academiccatalog.umd.edu"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; OrbitUMD/1.0; +mailto:jfrischm@terpmail.umd.edu)"
}
PROGRAMS_AZ_UNDERGRAD = "https://academiccatalog.umd.edu/undergraduate/colleges-schools/"

OUTPUT_PATH = "umd_requirements.jsonl"

def fetch_html(url: str) -> str:
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text

def make_soup(url: str) -> BeautifulSoup:
    html = fetch_html(url)
    return BeautifulSoup(html, "html.parser")

def get_undergrad_program_urls() -> list[dict]:
    soup = make_soup(PROGRAMS_AZ_UNDERGRAD)
    programs = []
    main = soup.find("div", class_="content") or soup
    for a in main.find_all("a", href=True):
        href = a["href"]
        text = a.get_text(strip=True)
        if not text or "Graduate" in text:
            continue
        if not href.startswith("/"):
            continue
        if not ("/undergraduate/" in href or "/colleges-schools/" in href):
            continue
        url = urljoin(BASE, href)
        programs.append({"name": text, "url": url})
    seen = set()
    deduped = []
    for p in programs:
        if p["url"] in seen:
            continue
        seen.add(p["url"])
        deduped.append(p)
    return deduped

def find_requirements_url(program_url: str):
    soup = make_soup(program_url)
    for a in soup.find_all("a", href=True):
        label = a.get_text(strip=True).lower()
        if label == "requirements":
            return urljoin(program_url, a["href"])
    return program_url

COURSE_CODE_RE = re.compile(r"\b[A-Z]{3,4}\s*\d{3}[A-Z]?\b")

def parse_course_list_table(table) -> list[dict]:
    rows = []
    for tr in table.find_all("tr"):
        cells = [c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])]
        if len(cells) < 2:
            continue
        if len(cells) == 2:
            course, title = cells
            credits = None
        else:
            course, title, credits = cells[0], cells[1], cells[2]
        if "course" in course.lower() and "title" in title.lower():
            continue
        rows.append(
            {
                "course_code": course,
                "title": title,
                "credits": credits,
                "raw_cells": cells,
            }
        )
    return rows

def extract_course_tables(requirements_soup: BeautifulSoup) -> list[dict]:
    result = []
    for table in requirements_soup.find_all("table"):
        header = table.find("tr")
        if not header:
            continue
        header_text = " ".join(
            c.get_text(" ", strip=True).lower()
            for c in header.find_all(["th", "td"])
        )
        if "course" in header_text and "title" in header_text:
            rows = parse_course_list_table(table)
            if rows:
                result.append(
                    {
                        "kind": "course_list_table",
                        "rows": rows,
                        "table_html": str(table),
                    }
                )
    return result

def parse_requirements_page(requirements_url: str) -> dict:
    soup = make_soup(requirements_url)
    h1 = soup.find("h1")
    title = h1.get_text(strip=True) if h1 else None
    course_blocks = extract_course_tables(soup)
    text_blocks = []
    content = soup.find("div", class_="content") or soup
    for node in content.find_all(["p", "li"]):
        text = node.get_text(" ", strip=True)
        if not text:
            continue
        if any(
            kw in text.lower()
            for kw in [
                "complete",
                "credits",
                "courses",
                "at least",
                "minimum",
                "must",
                "required",
                "track",
                "elective",
            ]
        ):
            text_blocks.append(text)
    return {
        "requirements_url": requirements_url,
        "page_title": title,
        "course_blocks": course_blocks,
        "text_blocks": text_blocks,
    }

def scrape_all_program_requirements(out_path: str = OUTPUT_PATH):
    programs = get_undergrad_program_urls()
    print(f"Found {len(programs)} candidate programs")
    with open(out_path, "w", encoding="utf-8") as f:
        for idx, prog in enumerate(programs, 1):
            name = prog["name"]
            url = prog["url"]
            try:
                req_url = find_requirements_url(url)
                if req_url is None:
                    print(f"[{idx}] {name}: no requirements tab found, skipping")
                    continue
                req_data = parse_requirements_page(req_url)
                obj = {
                    "program_name": name,
                    "program_url": url,
                    "requirements": req_data,
                }
                f.write(json.dumps(obj, ensure_ascii=False) + "\n")
                print(
                    f"[{idx}] scraped {name} "
                    f"(course blocks={len(req_data['course_blocks'])}, "
                    f"text rules={len(req_data['text_blocks'])})"
                )
            except Exception as e:
                print(f"[{idx}] ERROR for {name} ({url}): {e}")
            time.sleep(random.uniform(0.5, 1.5))

if __name__ == "__main__":
    scrape_all_program_requirements()


Found 632 candidate programs
[1] scraped Undergraduate (course blocks=0, text rules=14)
[2] scraped Undergraduate (course blocks=0, text rules=4)
[3] scraped Admissions Information (course blocks=0, text rules=6)
[4] scraped Office of Undergraduate Admissions (course blocks=0, text rules=4)
[5] scraped Freshman Admission (course blocks=0, text rules=13)
[6] scraped Transfer Admission (course blocks=0, text rules=8)
[7] scraped International Student Admission (course blocks=0, text rules=16)
[8] scraped Special Applicants (course blocks=0, text rules=12)
[9] scraped Admission to Limited Enrollment Programs (LEPs) (course blocks=0, text rules=7)
[10] scraped Extended Studies (course blocks=0, text rules=11)
[11] scraped Residency Information (course blocks=0, text rules=7)
[12] scraped Readmission and Reinstatement (course blocks=0, text rules=14)
[13] scraped Fees, Expenses and Financial Aid (course blocks=0, text rules=4)
[14] scraped Departmental Scholarships (course blocks=0, text ru

In [ ]:
import requests
from bs4 import BeautifulSoup
print("requests:", requests.__version__)
print("BeautifulSoup import works")